In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter  # ★平滑化フィルタをインポート
import copy
import warnings
warnings.filterwarnings('ignore')

# GPUが使えるか確認
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用デバイス: {device}")

base_path = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'

# ==========================================
# 1. 基準井戸の読み込みと波形マッチング（ヒント作成）
# ==========================================
print("1. 基準井戸の読み込みと特徴量エンジニアリング...")
typewell_files = glob.glob(f'{base_path}/**/*__typewell.csv', recursive=True)
typewells = {}
for f in typewell_files:
    well_id = os.path.basename(f).split('__')[0]
    df = pd.read_csv(f).sort_values('TVT').reset_index(drop=True)
    df['ref_GR_smooth'] = df['GR'].rolling(window=10, min_periods=1).mean()
    typewells[well_id] = df

def create_advanced_features(df, well_id):
    df['GR'] = df['GR'].ffill().fillna(0)
    df['Z'] = df['Z'].ffill().fillna(0)
    df = df.sort_values('MD').reset_index(drop=True)
    
    df['GR_smooth'] = df['GR'].rolling(window=10, min_periods=1).mean()
    df['GR_diff'] = df['GR'].diff().fillna(0)
    df['Z_diff'] = df['Z'].diff().fillna(0)
    
    if well_id in typewells:
        t_df = typewells[well_id].copy()
        t_df['ref_GR_smooth'] = t_df['ref_GR_smooth'].ffill().fillna(0)
        t_df_sorted = t_df[['ref_GR_smooth', 'TVT']].rename(columns={'TVT': 'matched_TVT'}).sort_values('ref_GR_smooth')
        
        merged = pd.merge_asof(
            df.sort_values('GR_smooth'),
            t_df_sorted,
            left_on='GR_smooth', right_on='ref_GR_smooth', direction='nearest'
        )
        df = merged.sort_values('MD').reset_index(drop=True).drop(columns=['ref_GR_smooth'])
    else:
        df['matched_TVT'] = df['Z']
    return df

train_files = glob.glob(f'{base_path}/train/*__horizontal_well.csv')
train_list = [create_advanced_features(pd.read_csv(f).assign(well_id=os.path.basename(f).split('__')[0]), os.path.basename(f).split('__')[0]) for f in train_files]
train_df = pd.concat(train_list, ignore_index=True)

test_files = glob.glob(f'{base_path}/test/*__horizontal_well.csv')
test_list = [create_advanced_features(pd.read_csv(f).assign(well_id=os.path.basename(f).split('__')[0], id=os.path.basename(f).split('__')[0] + '_' + pd.read_csv(f).index.astype(str)), os.path.basename(f).split('__')[0]) for f in test_files]
test_df = pd.concat(test_list, ignore_index=True)

features = ['MD', 'X', 'Y', 'Z', 'GR', 'GR_smooth', 'GR_diff', 'Z_diff', 'matched_TVT']
target = 'TVT'
train_df = train_df.dropna(subset=[target])

# 正規化
scaler = StandardScaler()
train_df[features] = scaler.fit_transform(train_df[features].fillna(0))
test_df[features] = scaler.transform(test_df[features].fillna(0))

# ==========================================
# 2. PyTorch用データセット
# ==========================================
WINDOW_SIZE = 15

class WellboreDataset(Dataset):
    def __init__(self, df, features, target_col=None, window_size=15):
        self.data = df[features].values
        self.targets = df[target_col].values if target_col else np.zeros(len(df))
        self.window = window_size
        self.length = len(df)
        self.padded_data = np.pad(self.data, ((self.window, self.window), (0, 0)), mode='edge')
        
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        seq = self.padded_data[idx : idx + 2 * self.window + 1].transpose(1, 0)
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(self.targets[idx], dtype=torch.float32)

# ==========================================
# 3. 1D-CNNモデルの定義
# ==========================================
class WaveformCNN(nn.Module):
    def __init__(self, num_features):
        super(WaveformCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv1d(in_channels=num_features, out_channels=64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        return self.fc_layers(x).squeeze()

# ==========================================
# 4. 学習ループ（最高精度版）
# ==========================================
print("2. 1D-CNNモデルの学習を開始します...")
gkf = GroupKFold(n_splits=5)
test_dataset = WellboreDataset(test_df, features, window_size=WINDOW_SIZE)

# 最も成績が良かった「バッチサイズ256」を復活
BATCH_SIZE = 256
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

preds = np.zeros(len(test_df))
EPOCHS = 30
PATIENCE = 5

for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, train_df[target], train_df['well_id'])):
    print(f"\n--- Fold {fold+1} ---")
    train_fold_df = train_df.iloc[train_idx].reset_index(drop=True)
    val_fold_df = train_df.iloc[val_idx].reset_index(drop=True)
    
    train_ds = WellboreDataset(train_fold_df, features, target_col=target, window_size=WINDOW_SIZE)
    val_ds = WellboreDataset(val_fold_df, features, target_col=target, window_size=WINDOW_SIZE)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    model = WaveformCNN(num_features=len(features)).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_val, y_val in val_loader:
                X_val, y_val = X_val.to(device), y_val.to(device)
                outputs = model(X_val)
                loss = criterion(outputs, y_val)
                val_loss += loss.item() * X_val.size(0)
        val_loss /= len(val_loader.dataset)
        
        print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"★ 早期終了 (Early Stopping)")
                break
                
    model.load_state_dict(best_model_state)
    
    model.eval()
    fold_preds = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            fold_preds.extend(outputs.cpu().numpy())
            
    preds += np.array(fold_preds) / gkf.n_splits

# ==========================================
# 5. 後処理（Savitzky-Golayフィルタ）と提出
# ==========================================
print("\n3. 予測結果に『物理的な平滑化（アイロンがけ）』を適用しています...")

test_df['predicted_tvt'] = preds

# ★ 井戸ごとに独立してフィルタをかける関数
def apply_savgol(x):
    # データ数が15より多い場合のみフィルタを適用
    if len(x) > 15:
        # 15個のデータを使って2次曲線でアイロンがけ
        return savgol_filter(x, window_length=15, polyorder=2)
    return x

# AIの生の予測を、物理的にあり得る滑らかな線に補正する
test_df['smoothed_tvt'] = test_df.groupby('well_id')['predicted_tvt'].transform(apply_savgol)

sub = pd.read_csv(f'{base_path}/sample_submission.csv')
# ここで生の predicted_tvt ではなく、アイロンがけ済みの smoothed_tvt を提出する
sub = sub.drop(columns=['tvt']).merge(test_df[['id', 'smoothed_tvt']], on='id', how='left')
sub = sub.rename(columns={'smoothed_tvt': 'tvt'})
sub['tvt'] = sub['tvt'].fillna(0.0)

sub[['id', 'tvt']].to_csv('submission.csv', index=False)
print("完了しました！純粋CNN ＋ 物理アイロンがけ版の submission.csv が作成されました。")